# 1. Data Collection

## Overview
This notebook scrapes ETF data from JustETF across four asset classes:

1. **Equities** — distributing UCITS ETFs scraped per country/region
2. **Bonds** — distributing UCITS bond ETFs scraped per currency/region
3. **Precious Metals** — physical gold/silver/platinum ETCs scraped globally (single CSV)
4. **Commodities** — broad commodity ETCs (crude oil, agriculture, metals) scraped globally (single CSV); screened at notebook 02 stage using InvestEngine availability

## Investment Universe Definition

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json

import pandas as pd
import numpy as np
import justetf_scraping

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.database import save_raw_etf_data

## Define Investment Universe
We'll start by creating a DataFrame of major economies, including their:
- Latest GDP (dynamically fetched from World Bank)
- MSCI market classification (Developed/Emerging)
- Geographic region
- Country code for data collection

This regional universe applies to **equities** and **bonds** only. Precious metals and commodities ETCs are sourced from a single global screen.

In [2]:
import requests
import pandas as pd
from datetime import datetime

# Fetch latest GDP data dynamically from the World Bank API
def fetch_gdp_data():
    current_year = datetime.now().year
    
    # We query the last 2 years (e.g., 2022:2024) to ensure we get the most recently available data
    # as some countries might not have reported the current year's GDP yet.
    # The World Bank API naturally returns the most recent non-null records first when queried by a date range.
    url = f"https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.CD?format=json&date={current_year-2}:{current_year}&per_page=1000"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        gdp_mapping = {}
        if len(data) > 1:
            for record in data[1]:
                country_id = record['country']['id']
                # If we haven't seen an entry for this country yet, and it's not None, add it.
                # Since the API returns data sorted descending by year, the first non-null is the most recent.
                if record.get('value') is not None and country_id not in gdp_mapping:
                    gdp_mapping[country_id] = {
                        'gdp': record['value'],
                        'year': record['date']
                    }
        return gdp_mapping
    except Exception as e:
        print(f"Error fetching GDP data: {e}")
        return {}

gdp_dict = fetch_gdp_data()

def format_gdp(val):
    if pd.isna(val) or val is None:
        return "N/A"
    if val >= 1e12:
        return f"${val/1e12:.3f} trillion"
    elif val >= 1e9:
        return f"${val/1e9:.3f} billion"
    else:
        return f"${val:,.0f}"

# Base country definitions without GDP
base_country_data = [
    ["United States", "Developed", "AmericasandUK", "US", "USD"],
    ["Canada", "Developed", "AmericasandUK", "CA", "CAD"],
    ["United Kingdom", "Developed", "AmericasandUK", "GB", "GBP"],
    ["Japan", "Developed", "APAC", "JP", "JPY"],
    ["Australia", "Developed", "APAC", "AU", "AUD"],
    ["Germany", "Developed", "EMEA", "DE", "EUR"],
    ["France", "Developed", "EMEA", "FR", "EUR"],
    ["Italy", "Developed", "EMEA", "IT", "EUR"],
    ["Spain", "Developed", "EMEA", "ES", "EUR"],
    ["Netherlands", "Developed", "EMEA", "NL", "EUR"],
    ["Switzerland", "Developed", "EMEA", "CH", "CHF"],
    ["Brazil", "Emerging", "Americas", "BR", "BRL"],
    ["Mexico", "Emerging", "Americas", "MX", "MXN"],
    ["China", "Emerging", "APACandEMEA", "CN", "CNY"],
    ["India", "Emerging", "APACandEMEA", "IN", "INR"],
    ["South Korea", "Emerging", "APACandEMEA", "KR", "KRW"],
    ["Indonesia", "Emerging", "APACandEMEA", "ID", "IDR"]
]

columns = ["Country", "MSCI", "Region", "Short_name", "Currency"]
country_df = pd.DataFrame(base_country_data, columns=columns)

# Map GDP dynamically
country_df["Latest GDP"] = country_df["Short_name"].apply(lambda x: gdp_dict.get(x, {}).get('gdp'))
country_df["GDP Year"] = country_df["Short_name"].apply(lambda x: gdp_dict.get(x, {}).get('year'))
country_df["Latest GDP"] = country_df["Latest GDP"].apply(format_gdp)

# Rename columns to reflect that it's the latest data
columns_updated = ["Country", "Latest GDP", "GDP Year", "MSCI", "Region", "Short_name", "Currency"]

# Reorder columns
country_df = country_df[columns_updated]

# Sort by MSCI then region
country_df.sort_values(by=["MSCI", "Region", "Latest GDP",], inplace=True)
country_df.reset_index(drop=True, inplace=True)

country_df



,Country,Latest GDP,GDP Year,MSCI,Region,Short_name,Currency
0,Australia,$1.757 trillion,2024,Developed,APAC,AU,AUD
1,Japan,$4.028 trillion,2024,Developed,APAC,JP,JPY
2,Canada,$2.244 trillion,2024,Developed,AmericasandUK,CA,CAD
3,United States,$28.751 trillion,2024,Developed,AmericasandUK,US,USD
4,United Kingdom,$3.686 trillion,2024,Developed,AmericasandUK,GB,GBP
5,Netherlands,$1.215 trillion,2024,Developed,EMEA,NL,EUR
6,Spain,$1.726 trillion,2024,Developed,EMEA,ES,EUR
7,Italy,$2.381 trillion,2024,Developed,EMEA,IT,EUR
8,France,$3.160 trillion,2024,Developed,EMEA,FR,EUR
9,Germany,$4.686 trillion,2024,Developed,EMEA,DE,EUR


## ETF Data Collection
Now we'll collect ETF data for all four asset classes:

- **Equities & Bonds** — scraped per country/region using the investment universe defined above
- **Precious Metals & Commodities** — scraped once globally (no regional breakdown)

Each scrape result is saved as a CSV in  and inserted into the  table in the SQLite database.

In [3]:
import traceback
import warnings

# Define asset classes to collect
# class-preciousMetals and class-commodities are scraped globally (no country loop)
asset_classes = [
    "class-equity",
    "class-bonds",
    "class-preciousMetals",
    "class-commodities",
]

# --- Global-only asset classes (no country/currency iteration) ---
GLOBAL_ASSET_CLASSES = {"class-preciousMetals", "class-commodities"}

# Group countries by region for data organization
region_countries = country_df.groupby('Region')['Short_name'].apply(list).to_dict()

# Create a reverse mapping of country to region
country_to_region = dict(zip(country_df['Short_name'], country_df['Region']))
country_to_country_name = dict(zip(country_df['Short_name'], country_df['Country']))
country_to_msci = dict(zip(country_df['Short_name'], country_df['MSCI']))

# Create a mapping of MSCI and Region to Category
msci_region_to_category = {
    ('Developed', 'AmericasandUK'): 'Developed_AmericasandUK',
    ('Developed', 'EMEA'): 'Developed_EMEA',
    ('Developed', 'APAC'): 'Developed_APAC',
    ('Emerging', 'Americas'): 'Emerging_Americas',
    ('Emerging', 'APACandEMEA'): 'Emerging_APACandEMEA'
}

for asset_class in asset_classes:
    asset_class_name = asset_class.split("-", 1)[1]

    # ------------------------------------------------------------------
    # Global-only asset classes: single scrape, no country iteration
    # ------------------------------------------------------------------
    if asset_class in GLOBAL_ASSET_CLASSES:
        try:
            df = justetf_scraping.load_overview(asset_class=asset_class, local_country="GB")
            df['region'] = 'Global'
            df['country'] = 'Global'
            df['region_category'] = 'Global'

            filename = f'justetf_class-{asset_class_name}_global.csv'
            filepath = DATA_RAW / filename
            df.to_csv(filepath, index=False)
            print(f"Scraped {len(df)} ETFs for {asset_class_name} (Global) → {filename}")
        except Exception as e:
            print(f"Error scraping {asset_class} (Global): {e}")
            continue

        continue  # skip the country loop below

    # ------------------------------------------------------------------
    # Country/currency iteration for equity and bonds
    # ------------------------------------------------------------------
    for country in country_df['Short_name']:
        # --- Scrape + CSV save (fatal on failure) ---
        try:
            if asset_class == "class-equity":
                df = justetf_scraping.load_overview(asset_class=asset_class, country=country, local_country="GB")
            else:
                currency = country_df[country_df['Short_name'] == country]['Currency'].values[0]
                df = justetf_scraping.load_overview(asset_class=asset_class, instrument_currency=currency, local_country="GB")

            df['region'] = country_to_region[country]
            df['country'] = country_to_country_name[country]

            msci = country_to_msci[country]
            region = country_to_region[country]
            region_category = msci_region_to_category.get((msci, region), "Unknown")
            df['region_category'] = region_category

            # Save CSV (append + dedup by ticker)
            filename = f'justetf_{asset_class}_{region_category.lower()}.csv'.lower()
            filepath = DATA_RAW / filename

            if filepath.exists():
                existing_df = pd.read_csv(filepath)
                combined_df = pd.concat([existing_df, df], ignore_index=True).drop_duplicates(subset=['ticker'])
                combined_df.to_csv(filepath, index=False)
            else:
                df.to_csv(filepath, index=False)

            print(f"Processed {country} data for {asset_class} -> {region_category}")
        except Exception as e:
            print(f"Error scraping {asset_class} for {country}: {e}")
            continue

# --- End of scraping ---
# Now, load all the accumulated CSVs and reliably write them to the DB once per category
print("\n--- Syncing collected CSVs to SQLite Database ---")
for filepath in sorted(DATA_RAW.glob("justetf_class-*.csv")):
    csvl = filepath.name
    ac = get_asset_class_from_filename(csvl)
    rc = get_region_category_from_filename(csvl)
    df_merged = pd.read_csv(filepath)
    try:
        save_raw_etf_data(df_merged, ac, rc)
        print(f"[DB] Synced {len(df_merged)} ETFs for {ac}/{rc}")
    except Exception as e:
        warnings.warn(f"[DB] Could not save {ac}/{rc} to database: {e}")

Processed AU data for class-equity -> Developed_APAC
Processed JP data for class-equity -> Developed_APAC
Processed CA data for class-equity -> Developed_AmericasandUK
Processed US data for class-equity -> Developed_AmericasandUK
Processed GB data for class-equity -> Developed_AmericasandUK
Processed NL data for class-equity -> Developed_EMEA
Processed ES data for class-equity -> Developed_EMEA
Processed IT data for class-equity -> Developed_EMEA
Processed FR data for class-equity -> Developed_EMEA
Processed DE data for class-equity -> Developed_EMEA
Processed CH data for class-equity -> Developed_EMEA
Processed ID data for class-equity -> Emerging_APACandEMEA
Processed KR data for class-equity -> Emerging_APACandEMEA
Processed CN data for class-equity -> Emerging_APACandEMEA
Processed IN data for class-equity -> Emerging_APACandEMEA
Processed MX data for class-equity -> Emerging_Americas
Processed BR data for class-equity -> Emerging_Americas
Processed AU data for class-bonds -> Devel

## Data Collection Summary
Display summary of collected ETFs by asset class and region:

In [4]:
asset_classes, country_df

(['class-equity', 'class-bonds', 'class-preciousMetals', 'class-commodities'],
            Country        Latest GDP GDP Year       MSCI         Region  \
 0        Australia   $1.757 trillion     2024  Developed           APAC   
 1            Japan   $4.028 trillion     2024  Developed           APAC   
 2           Canada   $2.244 trillion     2024  Developed  AmericasandUK   
 3    United States  $28.751 trillion     2024  Developed  AmericasandUK   
 4   United Kingdom   $3.686 trillion     2024  Developed  AmericasandUK   
 5      Netherlands   $1.215 trillion     2024  Developed           EMEA   
 6            Spain   $1.726 trillion     2024  Developed           EMEA   
 7            Italy   $2.381 trillion     2024  Developed           EMEA   
 8           France   $3.160 trillion     2024  Developed           EMEA   
 9          Germany   $4.686 trillion     2024  Developed           EMEA   
 10     Switzerland  $936.564 billion     2024  Developed           EMEA   
 11      

In [5]:
summary_data = []
for filepath in sorted(DATA_RAW.glob("justetf_class-*.csv")):
    csvl = filepath.name
    asset_class = get_asset_class_from_filename(csvl)
    region_category = get_region_category_from_filename(csvl)
    df = pd.read_csv(filepath)
    summary_data.append({
        'Asset Class': asset_class.title(),
        'Category': region_category,
        'NumberofETFs': len(df)
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df.pivot_table(
    index='Category',
    columns='Asset Class',
    values=['NumberofETFs'],
    aggfunc='first'
))

NumberofETFs                                  
Asset Class                    Bonds Commodities Equity Preciousmetals
Category                                                              
developed_americasanduk        555.0         NaN  698.0            NaN
developed_apac                  13.0         NaN  143.0            NaN
developed_emea                 441.0         NaN   88.0            NaN
emerging_americas                1.0         NaN    7.0            NaN
emerging_apacandemea            31.0         NaN   87.0            NaN
global                           NaN       121.0    NaN           52.0